# Load best checkpoint, inference, score, and LUM/HUM visualization

Notebook này dùng đúng setup của
`Train_full_mode_colab_run1_bs32_10epoch.ipynb`, nhưng **không train lại**.

Luồng chạy:

1. chuẩn bị repo/dependency/data giống notebook train;
2. tạo model đúng config `run1_bs32_10ep`;
3. load **EMA state** từ best checkpoint nếu checkpoint có EMA;
4. inference một sample và hiển thị xác suất 5 đáp án;
5. chạy lại score trên toàn bộ test split;
6. vẽ phân phối `U` và đường loss LUM/HUM.

> Lưu ý: LUM/HUM là hai công thức loss trong NCOD, không phải hai output
> score độc lập của model. `U` chỉ được học cho các sample thuộc train split.


## Kaggle runtime

Notebook này đọc dataset và checkpoint đã attach trực tiếp từ `/kaggle/input`; không mount Google Drive và không tải lại dữ liệu.


In [ ]:
# CELL 1: Repository Setup (Kaggle)
import os
from pathlib import Path

REPO_URL  = 'https://github.com/DanielQH07/tranSTR_Casual.git'
REPO_NAME = 'tranSTR_Casual'
BRANCH    = 'feat/generic-train-improvements'

WORKING_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
REPO_DIR    = WORKING_DIR / REPO_NAME

def has_repo_files(p):
    p = Path(p)
    return (p / 'DataLoader.py').exists() and (p / 'networks' / 'model.py').exists()

if has_repo_files(Path.cwd()):
    print(f'Using current repo: {Path.cwd()}')
else:
    if not REPO_DIR.exists():
        print(f'Cloning {REPO_URL} (branch={BRANCH})...')
        os.system(f'git clone {REPO_URL} -b {BRANCH} {REPO_DIR}')
    target_dir = REPO_DIR / 'causalvid'
    if target_dir.exists():
        os.chdir(target_dir)
    else:
        os.chdir(REPO_DIR)
    print(f'Changed directory to: {Path.cwd()}')

if not has_repo_files(Path.cwd()):
    raise FileNotFoundError(f'Repo files not found in {Path.cwd()}. Expected DataLoader.py and networks/model.py.')

print(f'Working directory: {Path.cwd()}')


In [ ]:
# CELL 2: Dependencies + W&B login (Kaggle Secrets / env)
print('=== CELL 2: Dependencies & W&B Setup ===')
import os, sys, importlib.util

os.system('pip install -q "huggingface_hub<1.0" "transformers>=4.41,<5.0" wandb peft kaggle kagglehub')

for mod in list(sys.modules):
    if mod.startswith('transformers') or mod.startswith('huggingface_hub'):
        del sys.modules[mod]

import huggingface_hub, transformers, wandb
print(f'huggingface_hub=={huggingface_hub.__version__} | transformers=={transformers.__version__}')

WANDB_PROJECT = 'transtr-causalvid-dino'
WANDB_ENTITY  = None

wandb_key = os.environ.get('WANDB_API_KEY')
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    try:
        wandb_key = wandb_key or _secrets.get_secret('WANDB_API_KEY')
    except Exception:
        pass
    print('Kaggle secrets checked')
except Exception as e:
    print(f'Kaggle secrets not available: {e}')

if wandb_key:
    wandb.login(key=wandb_key)
    print('W&B login OK')
else:
    os.environ.setdefault('WANDB_MODE', 'offline')
    print('WANDB_API_KEY not set; W&B will run offline')


In [ ]:
# CELL 3: Resolve attached Kaggle datasets (no download, no copy)
print('=== CELL 3: Resolve Kaggle Input Paths ===')

import os
from pathlib import Path

KAGGLE_INPUT = Path('/kaggle/input')
if not KAGGLE_INPUT.exists():
    raise RuntimeError('This inference notebook is configured for a Kaggle runtime.')

def _direct_file_count(path, suffix):
    try:
        return sum(1 for p in Path(path).iterdir() if p.is_file() and p.suffix == suffix)
    except OSError:
        return 0

def _find_data_dir(suffix=None, preferred_names=(), required_files=()):
    candidates = [KAGGLE_INPUT]
    candidates.extend(p for p in KAGGLE_INPUT.rglob('*') if p.is_dir())

    valid = []
    for path in candidates:
        if required_files and not all((path / name).exists() for name in required_files):
            continue
        count = _direct_file_count(path, suffix) if suffix else 1
        if suffix and count == 0:
            continue
        name_score = sum(name.lower() in str(path).lower() for name in preferred_names)
        valid.append((name_score, count, -len(path.parts), path))

    if not valid:
        return None
    return str(max(valid, key=lambda item: item[:3])[3])

CLIP_MERGED_PATH = _find_data_dir(
    suffix='.pt',
    preferred_names=('dinov3', 'merge', 'feature'),
)
GDINO_MERGED_PATH = _find_data_dir(
    suffix='.pkl',
    preferred_names=('gdinofrcnn', 'groundingdino', 'feature'),
)
ANNOTATION_QA_PATH = _find_data_dir(
    preferred_names=('text-annotation', 'QA'),
    required_files=('000000.json',),
)

# Annotation QA normally contains one directory per video, so use that shape
# when no marker file is available.
if ANNOTATION_QA_PATH is None:
    qa_dirs = [
        p for p in KAGGLE_INPUT.rglob('QA')
        if p.is_dir() and any(child.is_dir() for child in p.iterdir())
    ]
    ANNOTATION_QA_PATH = str(qa_dirs[0]) if qa_dirs else None

split_dirs = [
    p for p in KAGGLE_INPUT.rglob('*')
    if p.is_dir()
    and (p / 'train.pkl').exists()
    and (p / 'test.pkl').exists()
    and ((p / 'valid.pkl').exists() or (p / 'val.pkl').exists())
]
SPLIT_TXT_PATH = str(split_dirs[0]) if split_dirs else None

resolved = {
    'DINOv3 features': CLIP_MERGED_PATH,
    'GDINO+FRCNN features': GDINO_MERGED_PATH,
    'Annotations QA': ANNOTATION_QA_PATH,
    'Splits': SPLIT_TXT_PATH,
}
for name, path in resolved.items():
    print(f'{name}: {path or "NOT FOUND"}')

missing = [name for name, path in resolved.items() if path is None]
if missing:
    raise FileNotFoundError(
        'Missing attached Kaggle datasets: ' + ', '.join(missing)
        + '. Add them through Kaggle Notebook > Add Input.'
    )


In [ ]:
# CELL 4: Core imports + NCOD/HUM + Verifier/Knowledge training functions
print('=== CELL 4: Imports + Functions (NCOD + HUM + Verifier/Knowledge) ===')

import json
import math
import copy
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm

from utils.util import set_seed, set_gpu_devices
from DataLoader import VideoQADataset
from networks.model import VideoQAmodel

def _unpack_batch(batch):
    if len(batch) == 7:
        ff, of, q, a, ans_id, qns_key, q_family_id = batch
    elif len(batch) == 6:
        ff, of, q, a, ans_id, qns_key = batch
        q_family_id = None
    else:
        raise ValueError(f'Unexpected batch format with {len(batch)} elements')
    return ff, of, q, a, ans_id, qns_key, q_family_id

def _compute_sample_indices(qns_keys, key_to_idx, device):
    idxs = [key_to_idx.get(str(k), -1) for k in qns_keys]
    if any(i < 0 for i in idxs):
        missing = [str(qns_keys[i]) for i, v in enumerate(idxs) if v < 0][:5]
        raise KeyError(f'Missing qns_key in key_to_idx mapping: {missing}')
    return torch.tensor(idxs, dtype=torch.long, device=device)

_QTYPE_SUFFIXES = [
    'counterfactual_reason',
    'predictive_reason',
    'counterfactual',
    'predictive',
    'explanatory',
    'descriptive',
]

def _split_qns_key(qns_key):
    key = str(qns_key)
    for qtype in _QTYPE_SUFFIXES:
        suffix = f'_{qtype}'
        if key.endswith(suffix):
            return key[:-len(suffix)], qtype
    return key, 'unknown'

def _compute_acc_all_metrics(type_results):
    mapping = [
        ('Description', 'descriptive'),
        ('Explanation', 'explanatory'),
        ('Predictive-Answer', 'predictive'),
        ('Predictive-Reason', 'predictive_reason'),
        ('Counterfactual-Answer', 'counterfactual'),
        ('Counterfactual-Reason', 'counterfactual_reason'),
    ]
    metrics = {}
    for name, qtype in mapping:
        rows = type_results.get(qtype, [])
        metrics[name] = (sum(1 for r in rows if r['correct']) / len(rows) * 100) if rows else 0.0

    def _hard_metric(type_ans, type_reason):
        ans_by_vid = {r['video_id']: r['correct'] for r in type_results.get(type_ans, [])}
        reason_by_vid = {r['video_id']: r['correct'] for r in type_results.get(type_reason, [])}
        common_vids = set(ans_by_vid) & set(reason_by_vid)
        if not common_vids:
            return 0.0
        both_correct = sum(1 for vid in common_vids if ans_by_vid[vid] and reason_by_vid[vid])
        return both_correct / len(common_vids) * 100

    metrics['PAR'] = _hard_metric('predictive', 'predictive_reason')
    metrics['CAR'] = _hard_metric('counterfactual', 'counterfactual_reason')
    metrics['Acc_ALL'] = (metrics['Description'] + metrics['Explanation'] + metrics['PAR'] + metrics['CAR']) / 4.0
    return metrics

def _hard_negative_weights(cand_feat, target, enabled=False, max_weight=1.5):
    if (not enabled) or cand_feat is None or max_weight <= 1.0:
        return torch.ones_like(target, dtype=torch.float32)
    with torch.no_grad():
        cand_norm = F.normalize(cand_feat.detach(), dim=-1)
        gold = cand_norm.gather(1, target.view(-1, 1, 1).expand(-1, 1, cand_norm.size(-1))).squeeze(1)
        sims = torch.bmm(cand_norm, gold.unsqueeze(-1)).squeeze(-1)
        sims.scatter_(1, target.view(-1, 1), -1.0)
        hardness = sims.max(dim=1).values.clamp(min=0.0, max=1.0)
        return 1.0 + (float(max_weight) - 1.0) * hardness

def _update_ema_model(ema_model, model, decay):
    if ema_model is None:
        return
    with torch.no_grad():
        for ema_param, param in zip(ema_model.parameters(), model.parameters()):
            ema_param.data.mul_(decay).add_(param.data, alpha=1.0 - decay)
        for ema_buffer, buffer in zip(ema_model.buffers(), model.buffers()):
            ema_buffer.copy_(buffer)

def train_epoch_integrated(
    model, optimizer_model, optimizer_u, U, loader, xe, bce, device, epoch, key_to_idx,
    accumulation_steps=4, hum_alpha=1.0, lambda_verifier=0.2, lambda_knowledge=0.1,
    use_hard_neg_mining=False, hard_neg_weight_max=1.5,
    ema_model=None, ema_decay=0.999,
    scheduler=None, scheduler_step_per_batch=False
):
    model.train()
    total_loss, total_l1, total_l2 = 0.0, 0.0, 0.0
    total_verifier, total_knowledge = 0.0, 0.0
    correct, total = 0, 0
    optimizer_model.zero_grad()
    optimizer_u.zero_grad()

    pbar = tqdm(loader, desc=f'Epoch {epoch}', leave=False)
    for batch_idx, batch in enumerate(pbar):
        ff, of, q, a, ans_id, qns_keys, q_family_id = _unpack_batch(batch)
        ff, of, tgt = ff.to(device), of.to(device), ans_id.to(device)

        if q_family_id is None:
            q_family_id = torch.zeros_like(tgt)
        else:
            q_family_id = q_family_id.to(device)

        sample_indices = _compute_sample_indices(qns_keys, key_to_idx, device)
        out = model(ff, of, q, a, return_aux=True, q_family_id=q_family_id)
        logits = out['logits']
        fused_logits = out.get('fused_score', logits)
        verifier_logits = out.get('verifier_logits', logits)
        knowledge_logits = out.get('knowledge_score', None)
        cand_feat = out.get('cand_feat', None)

        probs = torch.softmax(logits, dim=1)
        y_onehot = F.one_hot(tgt, num_classes=logits.size(-1)).float()
        u_batch = U[sample_indices].unsqueeze(1)

        ce_per_sample = -torch.sum(y_onehot * torch.log(torch.clamp(probs, min=1e-12)), dim=1)
        shifted_probs = torch.clamp(probs + (u_batch.detach() * y_onehot), min=1e-12, max=1.0)
        lum_loss = -torch.sum(y_onehot * torch.log(shifted_probs), dim=1)
        hum_loss = (1.0 + hum_alpha * u_batch.detach().squeeze(1)) * ce_per_sample

        hard_neg_w = _hard_negative_weights(
            cand_feat,
            tgt,
            enabled=use_hard_neg_mining,
            max_weight=hard_neg_weight_max,
        )
        hard_mask = (q_family_id >= 2)
        l1_per_sample = torch.where(hard_mask, hum_loss, lum_loss)
        l1 = (l1_per_sample * hard_neg_w).mean()

        verifier_loss = bce(verifier_logits, y_onehot)
        if knowledge_logits is not None:
            knowledge_loss = bce(knowledge_logits, y_onehot)
        else:
            knowledge_loss = torch.tensor(0.0, device=device)

        model_loss = l1 + lambda_verifier * verifier_loss + lambda_knowledge * knowledge_loss
        (model_loss / accumulation_steps).backward()

        shifted_det = probs.detach() + (u_batch * y_onehot)
        l2 = F.mse_loss(shifted_det, y_onehot)
        (l2 / accumulation_steps).backward()

        if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer_model.step()
            _update_ema_model(ema_model, model, ema_decay)
            if scheduler is not None and scheduler_step_per_batch:
                scheduler.step()
            optimizer_model.zero_grad()
            optimizer_u.step()
            optimizer_u.zero_grad()
            with torch.no_grad():
                U.clamp_(0.0, 0.99)

        total_l1 += l1.item()
        total_l2 += l2.item()
        total_verifier += verifier_loss.item()
        total_knowledge += knowledge_loss.item()
        total_loss += (model_loss + l2).item()
        correct += (fused_logits.argmax(-1) == tgt).sum().item()
        total += tgt.size(0)

        pbar.set_postfix({
            'loss': total_loss / (batch_idx + 1),
            'l1': total_l1 / (batch_idx + 1),
            'l2': total_l2 / (batch_idx + 1),
            'ver': total_verifier / (batch_idx + 1),
            'know': total_knowledge / (batch_idx + 1),
            'acc': correct / max(total, 1) * 100
        })

    n = len(loader)
    return (
        total_loss / n,
        total_l1 / n,
        total_l2 / n,
        total_verifier / n,
        total_knowledge / n,
        correct / max(total, 1) * 100
    )

def eval_epoch(model, loader, device):
    model.eval()
    correct, total = 0, 0
    type_results = {}
    with torch.no_grad():
        for batch in loader:
            ff, of, q, a, ans_id, qns_keys, q_family_id = _unpack_batch(batch)
            ff, of, tgt = ff.to(device), of.to(device), ans_id.to(device)
            q_family_id = q_family_id.to(device) if q_family_id is not None else None
            out = model(ff, of, q, a, return_aux=True, q_family_id=q_family_id)
            logits = out.get('fused_score', out['logits'])
            preds = logits.argmax(-1)
            correct += (preds == tgt).sum().item()
            total += tgt.size(0)

            for key, pred, target in zip(qns_keys, preds.detach().cpu().tolist(), tgt.detach().cpu().tolist()):
                video_id, qtype = _split_qns_key(key)
                type_results.setdefault(qtype, []).append({
                    'video_id': video_id,
                    'correct': bool(int(pred) == int(target)),
                })

    metrics = _compute_acc_all_metrics(type_results)
    metrics['Plain_Acc'] = correct / max(total, 1) * 100
    return metrics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print('Imports and functions defined for integrated training.')

In [ ]:
# CELL 5: Setup Paths & Config (Kaggle inference)
print('=== CELL 5: Paths & Config ===')

clip_merged_path   = globals().get('CLIP_MERGED_PATH', None)
gdino_merged_path  = globals().get('GDINO_MERGED_PATH', None)
annotation_qa_path = globals().get('ANNOTATION_QA_PATH', None)
split_txt_path     = globals().get('SPLIT_TXT_PATH', None)

CLIP_FEATURE_PATH  = clip_merged_path  or '/kaggle/input/dinov3-feat'
GDINO_FEATURE_PATH = gdino_merged_path or '/kaggle/input/gdinofrcnn-features'
ANNOTATION_PATH    = annotation_qa_path or '/kaggle/input/text-annotation/QA'
SPLIT_DIR          = split_txt_path    or '/kaggle/input/casual-vid-data-split/split'

BASE = '/content/working' if os.path.exists('/content') else ('/kaggle/working' if os.path.exists('/kaggle/working') else os.getcwd())
MODEL_DIR = os.path.join(BASE, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

print('\n--- Path Verification ---')
def verify_path(name, path):
    if os.path.exists(path):
        items = os.listdir(path)[:3]
        print(f'OK {name}: {items}')
        return True
    print(f'NOT FOUND {name}: {path}')
    return False

all_ok = True
all_ok &= verify_path('DINOv3 Features (1024D)', CLIP_FEATURE_PATH)
all_ok &= verify_path('GDINO+FRCNN Features (2820D)', GDINO_FEATURE_PATH)
all_ok &= verify_path('Annotations (QA)', ANNOTATION_PATH)
all_ok &= verify_path('Splits', SPLIT_DIR)

if not all_ok:
    raise FileNotFoundError('One or more required data paths are missing. Re-run CELL 3 or attach Kaggle datasets.')

import glob as _glob
n_pt  = len(_glob.glob(os.path.join(CLIP_FEATURE_PATH, '*.pt')))
n_pkl = len(_glob.glob(os.path.join(GDINO_FEATURE_PATH, '*.pkl')))
print(f'\nDINOv3 .pt: {n_pt} | GDINO+FRCNN .pkl: {n_pkl}')

# ============================================
# 3-RUN TUNING PRESETS
# ============================================
RUN_TRAINING = True
RUN_PROFILE = 'run1'
RUN_VARIANT = 'colab_bs32_10ep_eslate'
RUN3_REG_MODE = 'dropout'

RUN_PROFILES = {
    'baseline': {
        'epoch': 10, 'lr': 1e-5,
        'lambda_verifier': 0.3, 'lambda_knowledge': 0.2,
        'early_stop_start_epoch': 8, 'early_stop_patience': 3,
        'dropout': 0.3, 'encoder_dropout': 0.3, 'decay': 1e-4,
    },
    'run1': {
        'epoch': 10, 'lr': 1e-5,
        'lambda_verifier': 0.25, 'lambda_knowledge': 0.3,
        'early_stop_start_epoch': 8, 'early_stop_patience': 3,
        'dropout': 0.3, 'encoder_dropout': 0.3, 'decay': 1e-4,
    },
    'run2': {
        'epoch': 10, 'lr': 8e-6,
        'lambda_verifier': 0.25, 'lambda_knowledge': 0.3,
        'early_stop_start_epoch': 8, 'early_stop_patience': 3,
        'dropout': 0.3, 'encoder_dropout': 0.3, 'decay': 1e-4,
    },
    'run3': {
        'epoch': 10, 'lr': 8e-6,
        'lambda_verifier': 0.25, 'lambda_knowledge': 0.3,
        'early_stop_start_epoch': 8, 'early_stop_patience': 3,
        'dropout': 0.3, 'encoder_dropout': 0.3, 'decay': 1e-4,
    },
}

if RUN_PROFILE not in RUN_PROFILES:
    raise ValueError(f'Invalid RUN_PROFILE={RUN_PROFILE}')

RUN_TAG = f'{RUN_PROFILE}_{RUN_VARIANT}'
MODEL_FILENAME = f'best_model_gdinofrcnn_ncod_hum_{RUN_TAG}.ckpt'
LATEST_CKPT_FILENAME = f'latest_checkpoint_gdinofrcnn_ncod_hum_{RUN_TAG}.ckpt'
TRAIN_HISTORY_FILENAME = f'train_history_gdinofrcnn_ncod_hum_{RUN_TAG}.csv'
PREDICTIONS_CSV_FILENAME = f'test_predictions_gdinofrcnn_ncod_hum_{RUN_TAG}.csv'
METRICS_JSON_FILENAME = f'final_metrics_gdinofrcnn_ncod_hum_{RUN_TAG}.json'
BEST_ARTIFACT_NAME = f'best-model-gdinofrcnn-ncod-hum-{RUN_TAG}'
LAST_ARTIFACT_NAME = f'last-checkpoint-gdinofrcnn-ncod-hum-{RUN_TAG}'
FINAL_ARTIFACT_NAME = f'final-results-gdinofrcnn-ncod-hum-{RUN_TAG}'

FEAT_DIM = 1024  # DINOv3 frame
OBJ_DIM  = 2820  # FRCNN(2048) + DeBERTa cls(768) + bbox(4)
OBJ_BBOX_DIM = 4
print(f'\nBackbone: DINOv3 ({FEAT_DIM}-d frame) + GroundingDINO+FRCNN ({OBJ_DIM}-d obj, bbox split={OBJ_BBOX_DIM})')
print(f'Run profile: {RUN_PROFILE}')

class Config:
    # Paths
    video_feature_root = CLIP_FEATURE_PATH
    grounding_dino_path = GDINO_FEATURE_PATH
    sample_list_path = ANNOTATION_PATH
    split_dir_txt = SPLIT_DIR

    # Model architecture
    topK_frame = 16
    objs = 12
    frames = 16
    select_frames = 5
    topK_obj = 12

    frame_feat_dim = FEAT_DIM
    obj_feat_dim = OBJ_DIM
    use_grounding_dino = True

    # GDINO object encoding fixes
    obj_use_bbox_pos_embed = True
    obj_bbox_dim = OBJ_BBOX_DIM
    obj_hard_gather_from_frame = True

    d_model = 768
    word_dim = 768
    nheads = 8
    num_encoder_layers = 2
    num_decoder_layers = 2
    normalize_before = True
    activation = 'gelu'
    dropout = 0.3
    encoder_dropout = 0.3

    # Text encoder
    text_encoder_type = 'microsoft/deberta-base'
    freeze_text_encoder = False
    text_encoder_lr = 5e-6
    text_pool_mode = 1
    use_lora = False
    lora_r = 8
    lora_alpha = 16
    lora_dropout = 0.1
    # DeBERTa v1 combines Q/K/V in a single attention projection.
    lora_target_modules = ['in_proj']

    # Training
    bs = 32
    accumulation_steps = 1
    lr = 1e-5
    epoch = 10
    gpu = 0
    gamma = 0.5
    decay = 1e-4
    n_query = 5
    lambda_verifier = 0.3
    lambda_knowledge = 0.2
    return_family_id = True

    # LR scheduler + early stopping
    lr_schedule = 'cosine_warmup'
    warmup_epochs = 1
    lr_patience = 1
    min_lr = 1e-7
    early_stop_patience = 3
    early_stop_min_delta = 0.02
    early_stop_start_epoch = 8

    # Generic training improvements
    use_hard_neg_mining = True
    hard_neg_weight_max = 1.5
    use_ema = True
    ema_decay = 0.999

    # NCOD + HUM
    ncod_u_lr = 0.1
    hum_alpha = 1.0

    # Aux warmup
    aux_warmup_epochs = 2

    # Other
    hard_eval = False
    pos_ratio = 1.0
    neg_ratio = 1.0
    a = 1.0
    num_workers = 4

for _k, _v in RUN_PROFILES[RUN_PROFILE].items():
    setattr(Config, _k, _v)

if RUN_PROFILE == 'run3':
    if RUN3_REG_MODE == 'dropout':
        Config.dropout = 0.25
        Config.encoder_dropout = 0.25
    elif RUN3_REG_MODE == 'decay':
        Config.decay = 8e-5
    else:
        raise ValueError(f'Invalid RUN3_REG_MODE={RUN3_REG_MODE}')

args = Config()

if args.text_encoder_type != 'microsoft/deberta-base':
    raise ValueError('Train notebook uses DeBERTa v1 only.')

set_gpu_devices(args.gpu)
set_seed(999)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')
print(f'Run tag: {RUN_TAG}')
print(f'Config: frame_feat_dim={args.frame_feat_dim}, obj_feat_dim={args.obj_feat_dim}, objs={args.objs}, select_frames={args.select_frames}')
print(f'use_grounding_dino={args.use_grounding_dino} -> obj_sorter SKIPPED')
print(f'obj_use_bbox_pos_embed={args.obj_use_bbox_pos_embed} | obj_hard_gather_from_frame={args.obj_hard_gather_from_frame}')
print(f'Effective bs: physical={args.bs} x accum={args.accumulation_steps} = {args.bs * args.accumulation_steps}')
print(f'lr={args.lr} | text_encoder_lr={args.text_encoder_lr} | decay={args.decay}')
print(f'use_lora={args.use_lora} | hard_neg={args.use_hard_neg_mining} (max={args.hard_neg_weight_max}) | ema={args.use_ema} (decay={args.ema_decay})')
print(f'lr_schedule={args.lr_schedule} | warmup_epochs={args.warmup_epochs}')
print(f'aux_warmup_epochs={args.aux_warmup_epochs} | verifier={args.lambda_verifier} | knowledge={args.lambda_knowledge}')
print(f'Early stop: start={args.early_stop_start_epoch}, patience={args.early_stop_patience}, min_delta={args.early_stop_min_delta}')
print(f'Model file: {MODEL_FILENAME}')

In [ ]:
# CELL 6: Create Datasets
print('=== CELL 6: Datasets ===')

train_ds = VideoQADataset(
    split='train', n_query=args.n_query, obj_num=args.objs,
    sample_list_path=args.sample_list_path,
    video_feature_path=args.video_feature_root,
    grounding_dino_path=args.grounding_dino_path,
    split_dir=args.split_dir_txt, topK_frame=args.topK_frame,
    max_samples=None, verbose=True, return_family_id=args.return_family_id
)
val_ds = VideoQADataset(
    split='val', n_query=args.n_query, obj_num=args.objs,
    sample_list_path=args.sample_list_path,
    video_feature_path=args.video_feature_root,
    grounding_dino_path=args.grounding_dino_path,
    split_dir=args.split_dir_txt, topK_frame=args.topK_frame,
    max_samples=None, verbose=True, return_family_id=args.return_family_id
)
test_ds = VideoQADataset(
    split='test', n_query=args.n_query, obj_num=args.objs,
    sample_list_path=args.sample_list_path,
    video_feature_path=args.video_feature_root,
    grounding_dino_path=args.grounding_dino_path,
    split_dir=args.split_dir_txt, topK_frame=args.topK_frame,
    max_samples=None, verbose=True, return_family_id=args.return_family_id
)

train_loader = DataLoader(train_ds, args.bs, shuffle=True, num_workers=args.num_workers, pin_memory=True)
val_loader = DataLoader(val_ds, args.bs, shuffle=False, num_workers=args.num_workers, pin_memory=True)
test_loader = DataLoader(test_ds, args.bs, shuffle=False, num_workers=args.num_workers, pin_memory=True)

train_sample_keys = [f"{row.video_id}_{row.type}" for row in train_ds.sample_list.itertuples(index=False)]
train_key_to_idx = {k: i for i, k in enumerate(train_sample_keys)}

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

## 1. Chọn và load best checkpoint

`CHECKPOINT_SOURCE="auto"` ưu tiên file local/Google Drive, sau đó mới tải
W&B artifact alias `best`. Có thể đặt trực tiếp `LOCAL_CHECKPOINT_PATH`.


In [ ]:
from pathlib import Path
import glob
import os
import copy
import torch
import wandb

CHECKPOINT_SOURCE = "auto"  # "auto", "local", hoặc "wandb"
LOCAL_CHECKPOINT_PATH = ""  # ví dụ: /kaggle/input/my-checkpoints/model.ckpt

# Có thể override bằng chuỗi đầy đủ entity/project/artifact:alias.
WANDB_BEST_ARTIFACT = ""

def resolve_best_checkpoint():
    if CHECKPOINT_SOURCE not in {"auto", "local", "wandb"}:
        raise ValueError(f"CHECKPOINT_SOURCE không hợp lệ: {CHECKPOINT_SOURCE}")

    local_candidates = []
    if LOCAL_CHECKPOINT_PATH:
        local_candidates.append(LOCAL_CHECKPOINT_PATH)
    local_candidates.extend([
        os.path.join(MODEL_DIR, MODEL_FILENAME),
        os.path.join("/kaggle/working/models", MODEL_FILENAME),
    ])
    local_candidates.extend(glob.glob(f"/kaggle/input/**/{MODEL_FILENAME}", recursive=True))
    local_candidates.extend(glob.glob("/kaggle/input/**/*.ckpt", recursive=True))

    if CHECKPOINT_SOURCE in {"auto", "local"}:
        for candidate in local_candidates:
            if candidate and os.path.isfile(candidate):
                print(f"Best checkpoint local: {candidate}")
                return candidate
        if CHECKPOINT_SOURCE == "local":
            raise FileNotFoundError(
                "Không tìm thấy local checkpoint. Hãy sửa LOCAL_CHECKPOINT_PATH."
            )

    api = wandb.Api()
    artifact_path = WANDB_BEST_ARTIFACT
    if not artifact_path:
        entity = WANDB_ENTITY or api.default_entity
        artifact_path = f"{entity}/{WANDB_PROJECT}/{BEST_ARTIFACT_NAME}:best"
    print(f"Downloading W&B artifact: {artifact_path}")
    artifact = api.artifact(artifact_path, type="model")
    artifact_dir = artifact.download(root="/kaggle/working/wandb_best_checkpoint")
    ckpt_files = sorted(Path(artifact_dir).glob("*.ckpt"))
    if not ckpt_files:
        raise FileNotFoundError(f"Artifact không chứa .ckpt: {artifact_dir}")
    return str(ckpt_files[0])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint_path = resolve_best_checkpoint()
checkpoint = torch.load(checkpoint_path, map_location=device)

cfg = {k: getattr(args, k) for k in Config.__dict__ if not k.startswith("_")}
cfg["device"] = device
cfg["topK_frame"] = args.select_frames
model = VideoQAmodel(**cfg).to(device)

ema_state = checkpoint.get("ema_model_state_dict")
state = ema_state if ema_state is not None else checkpoint["model_state_dict"]
missing, unexpected = model.load_state_dict(state, strict=False)
model.eval()

best_acc = float(checkpoint.get("best_acc", 0.0))
best_epoch = int(checkpoint.get("best_epoch", checkpoint.get("epoch", 0)))
print(f"Loaded: {checkpoint_path}")
print(f"State used: {'EMA' if ema_state is not None else 'model'}")
print(f"Best epoch: {best_epoch} | best val Acc_ALL: {best_acc:.2f}%")
print(f"Missing keys: {len(missing)} | unexpected keys: {len(unexpected)}")
if missing[:10]:
    print("Missing sample:", missing[:10])
if unexpected[:10]:
    print("Unexpected sample:", unexpected[:10])


## 2. Thống kê độ phức tạp mô hình

Cell này đo parameter count, kích thước weights/checkpoint, FLOPs được PyTorch profiler nhận diện, latency, throughput và peak GPU memory với batch size 1. FLOPs là **lower bound** vì dynamic top-K, tokenizer và một số custom attention operations có thể không được profiler tính đầy đủ.


In [ ]:
# Standardized model-complexity report for comparison with LLM/SOTA methods.
PROFILE_WARMUP_RUNS = 3
PROFILE_TIMED_RUNS = 10
PROFILE_SAMPLE_INDEX = 0

import os
import time
import numpy as np
import pandas as pd
from collections import defaultdict
from torch.profiler import profile, ProfilerActivity
from torch.utils.data import DataLoader, Subset

def _numel(parameters):
    return sum(parameter.numel() for parameter in parameters)

def _tensor_bytes(tensor):
    return tensor.numel() * tensor.element_size()

total_params = _numel(model.parameters())
trainable_params = _numel(
    parameter for parameter in model.parameters() if parameter.requires_grad
)
frozen_params = total_params - trainable_params
buffer_count = _numel(model.buffers())
parameter_bytes = sum(_tensor_bytes(parameter) for parameter in model.parameters())
buffer_bytes = sum(_tensor_bytes(buffer) for buffer in model.buffers())
state_dict_bytes = sum(_tensor_bytes(tensor) for tensor in model.state_dict().values())
checkpoint_bytes = os.path.getsize(checkpoint_path) if os.path.isfile(checkpoint_path) else 0

params_by_top_module = defaultdict(int)
trainable_by_top_module = defaultdict(int)
for name, parameter in model.named_parameters():
    top_module = name.split('.', 1)[0]
    params_by_top_module[top_module] += parameter.numel()
    if parameter.requires_grad:
        trainable_by_top_module[top_module] += parameter.numel()

module_rows = [
    {
        'module': module_name,
        'params_M': count / 1e6,
        'trainable_M': trainable_by_top_module[module_name] / 1e6,
        'share_percent': 100.0 * count / max(total_params, 1),
    }
    for module_name, count in sorted(
        params_by_top_module.items(),
        key=lambda item: item[1],
        reverse=True,
    )
]
module_profile_df = pd.DataFrame(module_rows)

profile_loader = DataLoader(
    Subset(test_ds, [PROFILE_SAMPLE_INDEX]),
    batch_size=1,
    shuffle=False,
    num_workers=0,
)
profile_batch = next(iter(profile_loader))
p_ff, p_of, p_q, p_a, _, _, p_family = _unpack_batch(profile_batch)
p_ff, p_of = p_ff.to(device), p_of.to(device)
p_family = p_family.to(device) if p_family is not None else None

def _profile_forward():
    return model(
        p_ff,
        p_of,
        p_q,
        p_a,
        return_aux=True,
        q_family_id=p_family,
    )

model.eval()
with torch.inference_mode():
    for _ in range(PROFILE_WARMUP_RUNS):
        _profile_forward()
    if device.type == 'cuda':
        torch.cuda.synchronize()

    activities = [ProfilerActivity.CPU]
    if device.type == 'cuda':
        activities.append(ProfilerActivity.CUDA)

    try:
        with profile(
            activities=activities,
            record_shapes=True,
            profile_memory=True,
            with_flops=True,
        ) as prof:
            _profile_forward()
            if device.type == 'cuda':
                torch.cuda.synchronize()
        counted_flops = sum(
            event.flops for event in prof.key_averages()
            if event.flops is not None
        )
        profiler_status = 'ok'
    except Exception as profiler_error:
        counted_flops = 0
        profiler_status = f'failed: {profiler_error}'

    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats(device)
        torch.cuda.synchronize()

    start_time = time.perf_counter()
    for _ in range(PROFILE_TIMED_RUNS):
        _profile_forward()
    if device.type == 'cuda':
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start_time

latency_ms = elapsed * 1000.0 / PROFILE_TIMED_RUNS
throughput_samples_per_sec = PROFILE_TIMED_RUNS / max(elapsed, 1e-12)
peak_gpu_allocated_bytes = (
    torch.cuda.max_memory_allocated(device) if device.type == 'cuda' else 0
)
peak_gpu_reserved_bytes = (
    torch.cuda.max_memory_reserved(device) if device.type == 'cuda' else 0
)

# For matmul/linear-heavy networks, one MAC is conventionally about two FLOPs.
counted_macs = counted_flops / 2.0
profile_summary = {
    'model': 'TranSTR-DN',
    'text_encoder': args.text_encoder_type,
    'use_lora': bool(args.use_lora),
    'input_frames': int(p_ff.shape[1]),
    'objects_per_frame': int(p_of.shape[2]),
    'answer_candidates': int(args.n_query),
    'selected_frames': int(args.select_frames),
    'total_params_M': total_params / 1e6,
    'trainable_params_M': trainable_params / 1e6,
    'frozen_params_M': frozen_params / 1e6,
    'trainable_percent': 100.0 * trainable_params / max(total_params, 1),
    'buffers_M': buffer_count / 1e6,
    'parameter_memory_MiB': parameter_bytes / (1024 ** 2),
    'state_dict_memory_MiB': state_dict_bytes / (1024 ** 2),
    'checkpoint_size_MiB': checkpoint_bytes / (1024 ** 2),
    'profiler_counted_GFLOPs_per_sample': counted_flops / 1e9,
    'profiler_counted_GMACs_per_sample': counted_macs / 1e9,
    'latency_ms_batch1': latency_ms,
    'throughput_samples_per_sec_batch1': throughput_samples_per_sec,
    'peak_gpu_allocated_MiB': peak_gpu_allocated_bytes / (1024 ** 2),
    'peak_gpu_reserved_MiB': peak_gpu_reserved_bytes / (1024 ** 2),
    'device': str(device),
    'dtype': str(next(model.parameters()).dtype),
    'flops_profiler_status': profiler_status,
}
model_complexity_df = pd.DataFrame([profile_summary])

print('=== Model Complexity Summary (batch size = 1) ===')
display(model_complexity_df.T.rename(columns={0: 'value'}))
print('=== Parameter Breakdown by Top-level Module ===')
display(
    module_profile_df.style.format({
        'params_M': '{:.3f}',
        'trainable_M': '{:.3f}',
        'share_percent': '{:.2f}%',
    })
)
print(
    'FLOPs note: profiler-counted FLOPs are a lower bound. '
    'Report the input protocol (16 frames, 12 objects/frame, 5 candidates) '
    'and hardware together with latency when comparing against SOTA/LLMs.'
)

MODEL_COMPLEXITY_CSV = os.path.join(MODEL_DIR, 'model_complexity_profile.csv')
MODULE_PARAMS_CSV = os.path.join(MODEL_DIR, 'model_parameter_breakdown.csv')
model_complexity_df.to_csv(MODEL_COMPLEXITY_CSV, index=False)
module_profile_df.to_csv(MODULE_PARAMS_CSV, index=False)
print(f'Saved: {MODEL_COMPLEXITY_CSV}')
print(f'Saved: {MODULE_PARAMS_CSV}')


## 3. Inference một sample


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader, Subset

SAMPLE_SPLIT = "test"  # "train", "val", hoặc "test"
SAMPLE_INDEX = 0

datasets = {"train": train_ds, "val": val_ds, "test": test_ds}
sample_ds = datasets[SAMPLE_SPLIT]
if not 0 <= SAMPLE_INDEX < len(sample_ds):
    raise IndexError(f"SAMPLE_INDEX phải thuộc [0, {len(sample_ds) - 1}]")

sample_loader = DataLoader(
    Subset(sample_ds, [SAMPLE_INDEX]),
    batch_size=1,
    shuffle=False,
    num_workers=0,
)
batch = next(iter(sample_loader))
ff, of, qns, ans_word, ans_id, qns_keys, q_family_id = _unpack_batch(batch)
ff, of = ff.to(device), of.to(device)
q_family_device = q_family_id.to(device) if q_family_id is not None else None

with torch.inference_mode():
    output = model(
        ff, of, qns, ans_word,
        return_aux=True,
        q_family_id=q_family_device,
    )
    logits = output.get("fused_score", output["logits"])
    probabilities = torch.softmax(logits, dim=-1)[0].cpu().numpy()

pred_idx = int(probabilities.argmax())
target_idx = int(ans_id[0])
row = sample_ds.sample_list.iloc[SAMPLE_INDEX]
answers = [str(row.get(f"a{i}", ans_word[i][0] if i < len(ans_word) else "")) for i in range(5)]
question = str(row.get("question", qns[0]))
qtype = str(row.get("type", "unknown"))
video_id = str(row.get("video_id", qns_keys[0]))
family_id = int(q_family_id[0]) if q_family_id is not None else 0

result_df = pd.DataFrame({
    "answer_idx": range(5),
    "answer": answers,
    "probability": probabilities,
    "is_prediction": [i == pred_idx for i in range(5)],
    "is_ground_truth": [i == target_idx for i in range(5)],
})

print(f"Split/index: {SAMPLE_SPLIT}/{SAMPLE_INDEX}")
print(f"Video: {video_id} | type: {qtype} | family_id: {family_id}")
print(f"Question: {question}")
print(f"Prediction: [{pred_idx}] {answers[pred_idx]}")
print(f"Ground truth: [{target_idx}] {answers[target_idx]}")
print(f"Correct: {pred_idx == target_idx}")
display(result_df.style.format({"probability": "{:.4f}"}))

plt.figure(figsize=(10, 4))
colors = [
    "#2ca02c" if i == target_idx else ("#d62728" if i == pred_idx else "#4c78a8")
    for i in range(5)
]
plt.bar([f"a{i}" for i in range(5)], probabilities, color=colors)
plt.ylim(0, max(1.0, float(probabilities.max()) * 1.15))
plt.ylabel("Probability")
plt.title("Single-sample answer probabilities")
for i, p in enumerate(probabilities):
    plt.text(i, p + 0.015, f"{p:.3f}", ha="center")
plt.show()


## 3. Chạy lại score trên test split


In [ ]:
from collections import defaultdict
from tqdm.auto import tqdm

QTYPE_SUFFIXES = [
    "counterfactual_reason", "predictive_reason", "counterfactual",
    "predictive", "explanatory", "descriptive",
]

def split_qns_key(qns_key):
    key = str(qns_key)
    for qtype_name in QTYPE_SUFFIXES:
        suffix = f"_{qtype_name}"
        if key.endswith(suffix):
            return key[:-len(suffix)], qtype_name
    return key, "unknown"

def compute_scores(type_results):
    mapping = [
        ("Description", "descriptive"),
        ("Explanation", "explanatory"),
        ("Predictive-Answer", "predictive"),
        ("Predictive-Reason", "predictive_reason"),
        ("Counterfactual-Answer", "counterfactual"),
        ("Counterfactual-Reason", "counterfactual_reason"),
    ]
    metrics = {}
    for metric_name, qtype_name in mapping:
        rows = type_results.get(qtype_name, [])
        metrics[metric_name] = (
            100.0 * sum(r["correct"] for r in rows) / len(rows) if rows else 0.0
        )

    def hard_metric(answer_type, reason_type):
        answer_by_video = {
            r["video_id"]: r["correct"] for r in type_results.get(answer_type, [])
        }
        reason_by_video = {
            r["video_id"]: r["correct"] for r in type_results.get(reason_type, [])
        }
        common = set(answer_by_video) & set(reason_by_video)
        if not common:
            return 0.0
        return 100.0 * sum(
            answer_by_video[v] and reason_by_video[v] for v in common
        ) / len(common)

    metrics["PAR"] = hard_metric("predictive", "predictive_reason")
    metrics["CAR"] = hard_metric("counterfactual", "counterfactual_reason")
    metrics["Acc_ALL"] = (
        metrics["Description"] + metrics["Explanation"]
        + metrics["PAR"] + metrics["CAR"]
    ) / 4.0
    metrics["WeightedScore_WeakPriority"] = (
        0.35 * metrics["Predictive-Reason"]
        + 0.35 * metrics["Counterfactual-Reason"]
        + 0.20 * metrics["Acc_ALL"]
        + 0.10 * best_acc
    )
    return metrics

def evaluate_test(model, loader):
    model.eval()
    type_results = defaultdict(list)
    prediction_rows = []
    correct_total = 0
    sample_total = 0

    with torch.inference_mode():
        for batch in tqdm(loader, desc="Scoring test"):
            ff, of, qns, ans_word, ans_id, qns_keys, q_family_id = _unpack_batch(batch)
            ff, of = ff.to(device), of.to(device)
            family = q_family_id.to(device) if q_family_id is not None else None
            output = model(ff, of, qns, ans_word, return_aux=True, q_family_id=family)
            logits = output.get("fused_score", output["logits"])
            probs = torch.softmax(logits, dim=-1).cpu()
            preds = probs.argmax(dim=-1)
            targets = ans_id.cpu()
            correct_total += int((preds == targets).sum())
            sample_total += len(targets)

            for key, pred, target, prob in zip(qns_keys, preds, targets, probs):
                video, qtype_name = split_qns_key(key)
                is_correct = int(pred) == int(target)
                type_results[qtype_name].append({
                    "video_id": video,
                    "correct": is_correct,
                })
                prediction_rows.append({
                    "qns_key": str(key),
                    "video_id": video,
                    "question_type": qtype_name,
                    "predicted_idx": int(pred),
                    "correct_idx": int(target),
                    "is_correct": int(is_correct),
                    "confidence": float(prob[int(pred)]),
                })

    metrics = compute_scores(type_results)
    metrics["Plain_Acc"] = 100.0 * correct_total / max(sample_total, 1)
    return metrics, pd.DataFrame(prediction_rows)

metrics, predictions_df = evaluate_test(model, test_loader)
score_df = pd.DataFrame(
    [{"metric": name, "score": value} for name, value in metrics.items()]
)
display(score_df.style.format({"score": "{:.2f}"}))

PREDICTIONS_OUTPUT = os.path.join(MODEL_DIR, f"reinference_{PREDICTIONS_CSV_FILENAME}")
METRICS_OUTPUT = os.path.join(MODEL_DIR, f"reinference_{METRICS_JSON_FILENAME}")
predictions_df.to_csv(PREDICTIONS_OUTPUT, index=False)
with open(METRICS_OUTPUT, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Predictions: {PREDICTIONS_OUTPUT}")
print(f"Metrics: {METRICS_OUTPUT}")

plot_order = [
    "Description", "Explanation", "Predictive-Answer", "Predictive-Reason",
    "Counterfactual-Answer", "Counterfactual-Reason", "PAR", "CAR",
    "Acc_ALL", "Plain_Acc",
]
plt.figure(figsize=(14, 5))
ax = sns.barplot(
    data=score_df[score_df.metric.isin(plot_order)],
    x="metric", y="score", order=plot_order, color="#4c78a8",
)
ax.set_ylim(0, 100)
ax.set_xlabel("")
ax.set_ylabel("Score (%)")
ax.tick_params(axis="x", rotation=35)
ax.set_title("Best-checkpoint test scores")
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f", padding=2)
plt.tight_layout()
plt.show()


## 4. Biểu đồ LUM và HUM

Với xác suất ground-truth `p_y` và uncertainty `u`:

- `LUM(u) = -log(clamp(p_y + u))`
- `HUM(u) = (1 + alpha * u) * (-log(p_y))`

Training chọn HUM khi `q_family_id >= 2`, ngược lại chọn LUM. Với sample test
không có `U` học riêng, notebook dùng median `U` của train làm mốc tham chiếu.


In [ ]:
U_checkpoint = checkpoint.get("U")
train_sample_keys_ckpt = checkpoint.get("train_sample_keys", [])
if U_checkpoint is None:
    raise KeyError("Checkpoint không chứa U; không thể vẽ chẩn đoán NCOD.")

u_values = U_checkpoint.detach().float().cpu().numpy().reshape(-1)
finite_u = u_values[np.isfinite(u_values)]
if finite_u.size == 0:
    raise ValueError("U trong checkpoint không có giá trị finite.")

selected_key = str(qns_keys[0])
key_to_u_idx = {str(key): i for i, key in enumerate(train_sample_keys_ckpt)}
if selected_key in key_to_u_idx and key_to_u_idx[selected_key] < len(u_values):
    selected_u = float(u_values[key_to_u_idx[selected_key]])
    u_source = "learned U của chính sample"
else:
    selected_u = float(np.median(finite_u))
    u_source = "median U train (sample hiện tại không thuộc train)"

p_gt = float(np.clip(probabilities[target_idx], 1e-12, 1.0))
alpha = float(args.hum_alpha)
q01, q99 = np.quantile(finite_u, [0.01, 0.99])
u_min = min(float(q01), selected_u, 0.0)
u_max = max(float(q99), selected_u, 0.0)
if np.isclose(u_min, u_max):
    u_min, u_max = u_min - 0.1, u_max + 0.1
padding = 0.08 * (u_max - u_min)
u_grid = np.linspace(u_min - padding, u_max + padding, 400)

ce = -np.log(p_gt)
lum_curve = -np.log(np.clip(p_gt + u_grid, 1e-12, 1.0))
hum_curve = (1.0 + alpha * u_grid) * ce
selected_lum = float(-np.log(np.clip(p_gt + selected_u, 1e-12, 1.0)))
selected_hum = float((1.0 + alpha * selected_u) * ce)
applied_mode = "HUM" if family_id >= 2 else "LUM"

print(f"p_ground_truth = {p_gt:.6f}")
print(f"U tham chiếu = {selected_u:.6f} ({u_source})")
print(f"LUM = {selected_lum:.6f} | HUM = {selected_hum:.6f}")
print(f"Mode áp dụng theo family_id={family_id}: {applied_mode}")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(finite_u, bins=50, kde=True, ax=axes[0], color="#4c78a8")
axes[0].axvline(selected_u, color="#d62728", linestyle="--", label=f"selected U={selected_u:.4f}")
axes[0].set_title("Checkpoint U distribution (train samples)")
axes[0].set_xlabel("U")
axes[0].legend()

axes[1].plot(u_grid, lum_curve, label="LUM", linewidth=2)
axes[1].plot(u_grid, hum_curve, label="HUM", linewidth=2)
axes[1].scatter([selected_u], [selected_lum], s=70, color="#1f77b4")
axes[1].scatter([selected_u], [selected_hum], s=70, color="#ff7f0e")
axes[1].axvline(selected_u, color="#777777", linestyle="--", alpha=0.7)
axes[1].set_title(f"LUM/HUM for selected sample - applied: {applied_mode}")
axes[1].set_xlabel("U")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 5. Kiểm tra các mẫu train có độ bất định cao

Cell dưới đây tìm đỉnh phụ ở phía phải của phân phối `U`, dùng đáy giữa hai đỉnh làm ngưỡng, rồi in annotation của các mẫu thuộc cụm high-uncertainty để kiểm tra nhiễu thủ công.


In [ ]:
# Detect the meaningful right-hand U peak and inspect its train samples.
HIGH_U_MAX_ROWS = 50
HIGH_U_HIST_BINS = 60
HIGH_U_MIN_PROMINENCE_RATIO = 0.03

from scipy.signal import find_peaks

finite_mask = np.isfinite(u_values)
finite_indices = np.flatnonzero(finite_mask)
finite_values = u_values[finite_mask]
hist, edges = np.histogram(finite_values, bins=HIGH_U_HIST_BINS)
centers = (edges[:-1] + edges[1:]) / 2.0

# Light smoothing prevents tiny tail fluctuations from being treated as modes.
kernel = np.array([1, 2, 3, 2, 1], dtype=float)
kernel /= kernel.sum()
smooth_hist = np.convolve(hist.astype(float), kernel, mode='same')
min_prominence = max(1.0, smooth_hist.max() * HIGH_U_MIN_PROMINENCE_RATIO)
peaks, properties = find_peaks(
    smooth_hist,
    prominence=min_prominence,
    distance=max(2, HIGH_U_HIST_BINS // 10),
)

if len(peaks) < 2:
    raise RuntimeError(
        f'Không tìm thấy hai đỉnh U có ý nghĩa (found={len(peaks)}). '
        'Thử giảm HIGH_U_MIN_PROMINENCE_RATIO hoặc tăng HIGH_U_HIST_BINS.'
    )

# Main mode is the tallest peak. Among modes to its right, select the most
# prominent one instead of an arbitrary tail fluctuation.
main_peak = int(peaks[np.argmax(smooth_hist[peaks])])
right_candidates = peaks[peaks > main_peak]
if len(right_candidates) == 0:
    raise RuntimeError('Có nhiều đỉnh U nhưng không có đỉnh phụ bên phải đỉnh chính.')
prominence_by_peak = dict(zip(peaks.tolist(), properties['prominences'].tolist()))
right_peak = int(max(right_candidates, key=lambda p: prominence_by_peak[int(p)]))
valley_rel = int(np.argmin(smooth_hist[main_peak:right_peak + 1]))
valley_idx = main_peak + valley_rel
high_u_threshold = float(edges[valley_idx + 1])

print(f'Đỉnh chính: U≈{centers[main_peak]:.6f}')
print(f'Đỉnh bất định bên phải: U≈{centers[right_peak]:.6f}')
print(f'Ngưỡng valley high-U: U >= {high_u_threshold:.6f}')

# Paper-ready U distribution: distinguish the regular mode from the
# high-uncertainty mode and mark the data-driven valley threshold.
PAPER_OUTPUT_DIR = Path('/kaggle/working/output')
PAPER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
U_DISTRIBUTION_JPG = PAPER_OUTPUT_DIR / 'u_distribution_paper.jpg'
U_DISTRIBUTION_VI_JPG = PAPER_OUTPUT_DIR / 'u_distribution_paper_vi.jpg'

high_u_count = int((finite_values >= high_u_threshold).sum())
regular_u_count = int(len(finite_values) - high_u_count)
high_u_percent = 100.0 * high_u_count / max(len(finite_values), 1)

fig, ax = plt.subplots(figsize=(10.5, 5.8))
bar_widths = np.diff(edges)
regular_bins = centers < high_u_threshold
high_bins = ~regular_bins
ax.bar(
    centers[regular_bins], hist[regular_bins], width=bar_widths[regular_bins],
    color='#4C78A8', edgecolor='white', linewidth=0.45, alpha=0.85,
    label=f'Regular uncertainty (n={regular_u_count:,})',
)
ax.bar(
    centers[high_bins], hist[high_bins], width=bar_widths[high_bins],
    color='#E45756', edgecolor='white', linewidth=0.45, alpha=0.88,
    label=f'High uncertainty (n={high_u_count:,}; {high_u_percent:.1f}%)',
)
ax.plot(
    centers, smooth_hist, color='#1F2937', linewidth=2.2,
    label='Smoothed frequency', zorder=4,
)
ax.axvline(
    high_u_threshold, color='#F59E0B', linestyle='--', linewidth=2.3,
    label=f'Valley threshold: U={high_u_threshold:.4f}', zorder=5,
)
ax.scatter(
    centers[main_peak], smooth_hist[main_peak], s=85,
    color='#2563EB', edgecolor='white', linewidth=1.0, zorder=6,
)
ax.scatter(
    centers[right_peak], smooth_hist[right_peak], s=85,
    color='#DC2626', edgecolor='white', linewidth=1.0, zorder=6,
)
main_mode_annotation = ax.annotate(
    f'Main mode\nU={centers[main_peak]:.4f}',
    xy=(centers[main_peak], smooth_hist[main_peak]), xytext=(10, 18),
    textcoords='offset points', ha='left', fontsize=10,
    arrowprops={'arrowstyle': '->', 'color': '#2563EB'},
)
high_mode_annotation = ax.annotate(
    f'High-U mode\nU={centers[right_peak]:.4f}',
    xy=(centers[right_peak], smooth_hist[right_peak]), xytext=(-12, 20),
    textcoords='offset points', ha='right', fontsize=10,
    arrowprops={'arrowstyle': '->', 'color': '#DC2626'},
)
ax.set_title('Distribution of Learned Sample Uncertainty on the Training Set', fontsize=14, pad=12)
ax.set_xlabel('Learned uncertainty variable $U_j$', fontsize=12)
ax.set_ylabel('Number of training samples', fontsize=12)
ax.tick_params(axis='both', labelsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.25)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(frameon=False, fontsize=9, loc='upper right')
fig.tight_layout()
fig.savefig(U_DISTRIBUTION_JPG, dpi=300, bbox_inches='tight', facecolor='white')

# Save an additional Vietnamese version for the thesis/paper.
ax.set_title('Phân phối độ bất định học được trên tập huấn luyện', fontsize=14, pad=12)
ax.set_xlabel('Biến bất định học được $U_j$', fontsize=12)
ax.set_ylabel('Số lượng mẫu huấn luyện', fontsize=12)
main_mode_annotation.set_text(f'Đỉnh chính\nU={centers[main_peak]:.4f}')
high_mode_annotation.set_text(f'Đỉnh bất định cao\nU={centers[right_peak]:.4f}')
legend_handles, _ = ax.get_legend_handles_labels()
ax.legend(
    legend_handles,
    [
        f'Bất định thông thường (n={regular_u_count:,})',
        f'Bất định cao (n={high_u_count:,}; {high_u_percent:.1f}%)',
        'Tần suất làm mượt',
        f'Ngưỡng đáy: U={high_u_threshold:.4f}',
    ],
    frameon=False, fontsize=9, loc='upper right',
)
fig.tight_layout()
fig.savefig(U_DISTRIBUTION_VI_JPG, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved English U distribution: {U_DISTRIBUTION_JPG}')
print(f'Saved Vietnamese U distribution: {U_DISTRIBUTION_VI_JPG}')

annotation_by_key = {
    f'{row.video_id}_{row.type}': row
    for row in train_ds.sample_list.itertuples(index=False)
}
high_positions = finite_indices[finite_values >= high_u_threshold]
high_positions = high_positions[np.argsort(u_values[high_positions])[::-1]]

review_rows = []
missing_annotation = 0
for u_idx in high_positions:
    if u_idx >= len(train_sample_keys_ckpt):
        continue
    qns_key = str(train_sample_keys_ckpt[u_idx])
    row = annotation_by_key.get(qns_key)
    if row is None:
        missing_annotation += 1
        continue
    target_idx = int(row.answer)
    answers = [str(getattr(row, f'a{i}')) for i in range(5)]
    review_rows.append({
        'U': float(u_values[u_idx]),
        'qns_key': qns_key,
        'video_id': str(row.video_id),
        'question_type': str(row.type),
        'question': str(row.question),
        'correct_idx': target_idx,
        'correct_answer': answers[target_idx] if 0 <= target_idx < len(answers) else '',
        'a0': answers[0], 'a1': answers[1], 'a2': answers[2],
        'a3': answers[3], 'a4': answers[4],
    })

high_u_review_df = pd.DataFrame(review_rows).sort_values('U', ascending=False)
HIGH_U_REVIEW_CSV = os.path.join(MODEL_DIR, 'high_uncertainty_train_samples.csv')
high_u_review_df.to_csv(HIGH_U_REVIEW_CSV, index=False)

print(f'High-U samples matched to train annotations: {len(high_u_review_df)}')
print(f'Missing annotation keys: {missing_annotation}')
print(f'Saved full review list: {HIGH_U_REVIEW_CSV}')
display_cols = [
    'U', 'qns_key', 'question_type', 'question', 'correct_idx',
    'correct_answer', 'a0', 'a1', 'a2', 'a3', 'a4',
]
display(high_u_review_df[display_cols].head(HIGH_U_MAX_ROWS))


## 6. Xem video và các bước diễn biến của mẫu nghi nhiễu

Attach dataset chứa video `.mp4` vào Kaggle, sau đó chọn `NOISY_SAMPLE_ROW`. Cell sẽ lấy frame đều theo thời gian để đối chiếu diễn biến video với loại câu hỏi và annotation.


In [ ]:
NOISY_SAMPLE_ROWS = [0, 1, 2]  # Các dòng high-U cần kiểm tra
VIDEO_FRAME_SAMPLES = 16       # Số frame tổng quan cho mỗi video
DISPLAY_FULL_VIDEO = False     # Nên False khi chạy nhiều mẫu
VIDEO_ROOT_OVERRIDE = '/kaggle/input/datasets/danielq07/causal-vidqa-raw-video-full'

import cv2
import pickle
import re
from matplotlib.patches import Rectangle
from torch.utils.data import DataLoader, Subset
from IPython.display import display, Video

def _normalize_video_stem(value):
    return Path(str(value)).stem.lower()

def _build_video_index(root):
    index = {}
    supported = {'.mp4', '.avi', '.mov', '.mkv'}
    for video_path in Path(root).rglob('*'):
        if video_path.is_file() and video_path.suffix.lower() in supported:
            index.setdefault(_normalize_video_stem(video_path.name), str(video_path))
    return index

video_search_root = Path(VIDEO_ROOT_OVERRIDE) if VIDEO_ROOT_OVERRIDE else Path('/kaggle/input')
if not video_search_root.exists():
    raise FileNotFoundError(f'Video search root does not exist: {video_search_root}')

video_index = _build_video_index(video_search_root)
print(f'Indexed video files: {len(video_index)} from {video_search_root}')

def _resolve_video_path(video_id):
    key = _normalize_video_stem(video_id)
    if key in video_index:
        return video_index[key]
    matches = [
        video_path for stem, video_path in video_index.items()
        if stem.endswith(key) or key.endswith(stem)
    ]
    return matches[0] if len(matches) == 1 else None

def _sample_video_frames(video_path, num_samples=12):
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise RuntimeError(f'Cannot open video: {video_path}')

    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = float(capture.get(cv2.CAP_PROP_FPS))
    fps = fps if np.isfinite(fps) and fps > 0 else 1.0
    if frame_count <= 0:
        capture.release()
        raise RuntimeError(f'Invalid frame count for video: {video_path}')

    frame_indices = np.linspace(
        0, frame_count - 1, min(num_samples, frame_count)
    ).astype(int)
    sampled = []
    for step, frame_idx in enumerate(frame_indices, start=1):
        capture.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ok, frame_bgr = capture.read()
        if not ok:
            continue
        sampled.append({
            'step': step,
            'frame_idx': int(frame_idx),
            'time_sec': float(frame_idx / fps),
            'image': cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB),
        })
    capture.release()
    return sampled, frame_count, fps


def inspect_noisy_sample(row_index):
    if high_u_review_df.empty:
        raise RuntimeError('high_u_review_df is empty; run the high-U detection cell first.')
    if not 0 <= row_index < len(high_u_review_df):
        raise IndexError(f'row_index must be in [0, {len(high_u_review_df) - 1}]')

    noisy_sample = high_u_review_df.iloc[row_index]
    print('\n' + '=' * 100)
    print(f'Inspecting high-U row {row_index}')
    selected_video_path = _resolve_video_path(noisy_sample['video_id'])
    if selected_video_path is None:
        raise FileNotFoundError(
            f"Cannot match video_id={noisy_sample['video_id']} under {video_search_root}. "
            'Set VIDEO_ROOT_OVERRIDE to the attached video directory if needed.'
        )

    print(f"U: {noisy_sample['U']:.6f}")
    print(f"qns_key: {noisy_sample['qns_key']}")
    print(f"Video: {selected_video_path}")
    print(f"Question type: {noisy_sample['question_type']}")
    print(f"Question: {noisy_sample['question']}")
    print(f"Annotated answer: [{int(noisy_sample['correct_idx'])}] {noisy_sample['correct_answer']}")

    safe_qns_key = re.sub(r'[^A-Za-z0-9_.-]+', '_', str(noisy_sample['qns_key']))
    VIDEO_ARTIFACT_DIR = Path('/kaggle/working/output') / f'noisy_sample_{safe_qns_key}'
    VIDEO_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

    answers_df = pd.DataFrame({
        'answer_idx': range(5),
        'answer': [noisy_sample[f'a{i}'] for i in range(5)],
        'is_annotated_correct': [
            i == int(noisy_sample['correct_idx']) for i in range(5)
        ],
    })
    display(answers_df)
    answers_df.to_csv(VIDEO_ARTIFACT_DIR / 'question_answers.csv', index=False)

    qa_text = (
        f"U: {noisy_sample['U']:.6f}\n"
        f"Question type: {noisy_sample['question_type']}\n"
        f"Question: {noisy_sample['question']}\n\n"
        + '\n'.join(
            f"[{i}] {noisy_sample[f'a{i}']}"
            + ('  <-- annotated answer' if i == int(noisy_sample['correct_idx']) else '')
            for i in range(5)
        )
    )
    (VIDEO_ARTIFACT_DIR / 'question_answers.txt').write_text(qa_text, encoding='utf-8')
    qa_fig, qa_ax = plt.subplots(figsize=(12, 6))
    qa_ax.axis('off')
    qa_ax.text(0.02, 0.98, qa_text, va='top', ha='left', fontsize=12, wrap=True)
    qa_fig.tight_layout()
    qa_fig.savefig(
        VIDEO_ARTIFACT_DIR / 'question_answers.jpg',
        dpi=250, bbox_inches='tight', facecolor='white',
    )
    plt.show()

    sampled_frames, source_frame_count, source_fps = _sample_video_frames(
        selected_video_path,
        VIDEO_FRAME_SAMPLES,
    )
    if not sampled_frames:
        raise RuntimeError(f'No frames could be decoded from: {selected_video_path}')

    duration_sec = source_frame_count / source_fps
    print(
        f'Video metadata: {source_frame_count} frames | '
        f'{source_fps:.2f} FPS | {duration_sec:.2f}s'
    )

    cols = 4
    rows = int(np.ceil(len(sampled_frames) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(16, 3.6 * rows))
    axes = np.atleast_1d(axes).reshape(-1)
    for ax, item in zip(axes, sampled_frames):
        ax.imshow(item['image'])
        ax.set_title(
            f"Step {item['step']}/{len(sampled_frames)} | "
            f"t={item['time_sec']:.2f}s | frame={item['frame_idx']}"
        )
        ax.axis('off')
    for ax in axes[len(sampled_frames):]:
        ax.axis('off')
    fig.suptitle(
        f"High-U sample ({noisy_sample['question_type']}) | "
        f"U={noisy_sample['U']:.6f}\n{noisy_sample['question']}",
        fontsize=13,
    )
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    OVERVIEW_FRAMES_JPG = VIDEO_ARTIFACT_DIR / 'overview_uniform_frames.jpg'
    fig.savefig(OVERVIEW_FRAMES_JPG, dpi=250, bbox_inches='tight', facecolor='white')
    for item in sampled_frames:
        frame_file = VIDEO_ARTIFACT_DIR / f"overview_step_{item['step']:02d}_frame_{item['frame_idx']:06d}.jpg"
        cv2.imwrite(str(frame_file), cv2.cvtColor(item['image'], cv2.COLOR_RGB2BGR))
    plt.show()

    # Capture the exact five frame slots selected by the model for this sample.
    train_key_to_dataset_idx = {
        f'{row.video_id}_{row.type}': idx
        for idx, row in enumerate(train_ds.sample_list.itertuples(index=False))
    }
    selected_dataset_idx = train_key_to_dataset_idx.get(str(noisy_sample['qns_key']))
    if selected_dataset_idx is None:
        raise KeyError(f"Cannot find train sample: {noisy_sample['qns_key']}")

    selected_loader = DataLoader(
        Subset(train_ds, [selected_dataset_idx]),
        batch_size=1,
        shuffle=False,
        num_workers=0,
    )
    selected_batch = next(iter(selected_loader))
    sel_ff, sel_of, sel_q, sel_a, _, _, sel_family = _unpack_batch(selected_batch)
    sel_ff, sel_of = sel_ff.to(device), sel_of.to(device)
    sel_family = sel_family.to(device) if sel_family is not None else None

    # PerturbedTopK overrides __call__ instead of forward, so normal PyTorch
    # forward hooks are bypassed. Reproduce the exact frame-selection stages here.
    FRAME_SELECTION_SEED = 999
    feature_frame_count = int(sel_ff.shape[1])
    with torch.inference_mode():
        resized_frames = model.frame_resize(sel_ff)
        question_tokens, question_mask = model.forward_text(list(sel_q), device)
        valid_frame_mask = torch.ones(
            sel_ff.shape[0],
            feature_frame_count,
            dtype=torch.bool,
            device=device,
        )
        _, frame_attention = model.frame_decoder(
            resized_frames,
            question_tokens,
            memory_key_padding_mask=question_mask,
            query_pos=model.pos_encoder_1d(valid_frame_mask, model.d_model),
            output_attentions=True,
        )

        # PerturbedTopK samples Gaussian noise even in eval mode. Forking the RNG
        # makes this visualization repeatable without changing global RNG state.
        fork_devices = [device.index or 0] if device.type == 'cuda' else []
        with torch.random.fork_rng(devices=fork_devices):
            torch.manual_seed(FRAME_SELECTION_SEED)
            if device.type == 'cuda':
                torch.cuda.manual_seed_all(FRAME_SELECTION_SEED)
            sorter_output = model.frame_sorter(frame_attention.flatten(1, 2))

    # sorter_output is [B, F*Q, K]. This matches model.forward:
    # rearrange(..., 'b (f q) k -> b f q k', f=F).sum(-2).
    frame_weights = sorter_output.reshape(
        sorter_output.shape[0],
        feature_frame_count,
        -1,
        sorter_output.shape[-1],
    ).sum(dim=2)
    selected_feature_slots = (
        frame_weights[0].argmax(dim=0).cpu().numpy().astype(int)
    )
    print(
        f'Selected input-frame slots (seed={FRAME_SELECTION_SEED}): '
        f'{selected_feature_slots.tolist()}'
    )

    gdino_pkl_path = Path(GDINO_FEATURE_PATH) / f"{noisy_sample['video_id']}.pkl"
    if not gdino_pkl_path.exists():
        matches = list(Path(GDINO_FEATURE_PATH).rglob(f"{noisy_sample['video_id']}.pkl"))
        gdino_pkl_path = matches[0] if matches else gdino_pkl_path
    if not gdino_pkl_path.exists():
        raise FileNotFoundError(f'GroundingDINO metadata not found: {gdino_pkl_path}')

    with open(gdino_pkl_path, 'rb') as file:
        gdino_package = pickle.load(file)

    frames_data = gdino_package.get('frames', [])
    if not frames_data:
        raise RuntimeError(f'No frames metadata in: {gdino_pkl_path}')

    # Reproduce DataLoader's alignment from all GDINO frames to the 16 model inputs.
    if len(frames_data) > feature_frame_count:
        gdino_sample_positions = np.linspace(
            0, len(frames_data) - 1, feature_frame_count
        ).astype(int)
    else:
        gdino_sample_positions = np.arange(len(frames_data), dtype=int)
        if len(gdino_sample_positions) < feature_frame_count:
            gdino_sample_positions = np.pad(
                gdino_sample_positions,
                (0, feature_frame_count - len(gdino_sample_positions)),
                mode='edge',
            )

    def _read_raw_frame(video_path, frame_idx):
        capture = cv2.VideoCapture(str(video_path))
        capture.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ok, frame_bgr = capture.read()
        capture.release()
        if not ok:
            return None
        return cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

    def _draw_gdino_boxes(frame_rgb, frame_meta, frame_label):
        annotated = frame_rgb.copy()
        height, width = annotated.shape[:2]
        boxes = np.asarray(
            frame_meta.get('boxes_xyxy_orig', np.zeros((0, 4))),
            dtype=float,
        )
        labels = list(frame_meta.get('labels_text', []))
        valid_box_count = 0

        # Header makes the temporal order explicit in every downloaded image.
        cv2.rectangle(annotated, (0, 0), (width, 42), (20, 20, 20), -1)
        cv2.putText(
            annotated, frame_label, (12, 29),
            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA,
        )

        for box_idx, box in enumerate(boxes[:args.objs]):
            if len(box) < 4 or not np.all(np.isfinite(box)):
                continue
            x1, y1, x2, y2 = map(float, box[:4])
            x1, x2 = np.clip([x1, x2], 0, width - 1)
            y1, y2 = np.clip([y1, y2], 43, height - 1)
            if x2 <= x1 or y2 <= y1:
                continue
            label = str(labels[box_idx]) if box_idx < len(labels) else f'obj_{box_idx}'
            color_rgb = tuple(
                int(255 * channel) for channel in plt.cm.tab20(box_idx % 20)[:3]
            )
            cv2.rectangle(
                annotated,
                (int(x1), int(y1)), (int(x2), int(y2)), color_rgb, 3,
            )
            text_y = max(60, int(y1) - 6)
            cv2.putText(
                annotated, label, (int(x1), text_y),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color_rgb, 2, cv2.LINE_AA,
            )
            valid_box_count += 1
        return annotated, valid_box_count, labels

    # Save all 16 model-input frames with GroundingDINO boxes and class names.
    ALL_16_BOXED_DIR = VIDEO_ARTIFACT_DIR / 'all_16_frames_with_boxes'
    ALL_16_BOXED_DIR.mkdir(parents=True, exist_ok=True)
    all_frame_rows = []
    for feature_slot, gdino_position in enumerate(gdino_sample_positions):
        frame_meta = frames_data[int(gdino_position)]
        raw_frame_idx = int(frame_meta.get('frame_idx', feature_slot))
        time_sec = raw_frame_idx / source_fps
        frame_rgb = _read_raw_frame(selected_video_path, raw_frame_idx)
        if frame_rgb is None:
            print(f'Warning: failed to decode input frame {feature_slot + 1}/16')
            continue

        frame_label = (
            f'Frame {feature_slot + 1:02d}/{feature_frame_count:02d} | '
            f'raw={raw_frame_idx} | t={time_sec:.2f}s'
        )
        annotated_frame, box_count, frame_labels = _draw_gdino_boxes(
            frame_rgb,
            frame_meta,
            frame_label,
        )
        frame_output_path = ALL_16_BOXED_DIR / (
            f'frame_{feature_slot + 1:02d}_of_{feature_frame_count:02d}'
            f'_raw_{raw_frame_idx:06d}.jpg'
        )
        cv2.imwrite(
            str(frame_output_path),
            cv2.cvtColor(annotated_frame, cv2.COLOR_RGB2BGR),
        )
        all_frame_rows.append({
            'frame_order': feature_slot + 1,
            'input_frame_slot': feature_slot,
            'gdino_position': int(gdino_position),
            'raw_frame_idx': raw_frame_idx,
            'time_sec': time_sec,
            'box_count': box_count,
            'labels': ', '.join(map(str, frame_labels[:args.objs])),
            'image_path': str(frame_output_path),
        })

    all_16_frames_df = pd.DataFrame(all_frame_rows)
    all_16_frames_df.to_csv(
        VIDEO_ARTIFACT_DIR / 'all_16_frames_with_boxes_metadata.csv',
        index=False,
    )
    print(f'Saved {len(all_16_frames_df)} boxed input frames to: {ALL_16_BOXED_DIR}')

    selected_frame_metadata = []
    for rank, feature_slot in enumerate(selected_feature_slots, start=1):
        gdino_position = int(gdino_sample_positions[int(feature_slot)])
        frame_meta = frames_data[gdino_position]
        raw_frame_idx = int(frame_meta.get('frame_idx', feature_slot))
        selected_frame_metadata.append({
            'rank': rank,
            'feature_slot': int(feature_slot),
            'gdino_position': gdino_position,
            'raw_frame_idx': raw_frame_idx,
            'weight': float(frame_weights[0, feature_slot, rank - 1].cpu()),
            'boxes': np.asarray(
                frame_meta.get('boxes_xyxy_orig', np.zeros((0, 4))),
                dtype=float,
            ),
            'labels': list(frame_meta.get('labels_text', [])),
        })

    fig, axes = plt.subplots(1, len(selected_frame_metadata), figsize=(25, 5.5))
    axes = np.atleast_1d(axes)
    selected_rows = []
    for ax, item in zip(axes, selected_frame_metadata):
        frame_rgb = _read_raw_frame(selected_video_path, item['raw_frame_idx'])
        if frame_rgb is None:
            ax.set_title(f"Selected {item['rank']}: decode failed")
            ax.axis('off')
            continue

        annotated_frame = frame_rgb.copy()
        height, width = annotated_frame.shape[:2]
        valid_box_count = 0
        for box_idx, box in enumerate(item['boxes'][:args.objs]):
            if len(box) < 4 or not np.all(np.isfinite(box)):
                continue
            x1, y1, x2, y2 = map(float, box[:4])
            x1, x2 = np.clip([x1, x2], 0, width - 1)
            y1, y2 = np.clip([y1, y2], 0, height - 1)
            if x2 <= x1 or y2 <= y1:
                continue
            label = (
                str(item['labels'][box_idx])
                if box_idx < len(item['labels'])
                else f'obj_{box_idx}'
            )
            color_rgb = tuple(
                int(255 * channel) for channel in plt.cm.tab20(box_idx % 20)[:3]
            )
            cv2.rectangle(
                annotated_frame,
                (int(x1), int(y1)), (int(x2), int(y2)), color_rgb, 3,
            )
            text_y = max(18, int(y1) - 6)
            cv2.putText(
                annotated_frame, label, (int(x1), text_y),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color_rgb, 2, cv2.LINE_AA,
            )
            valid_box_count += 1

        ax.imshow(annotated_frame)
        selected_frame_file = VIDEO_ARTIFACT_DIR / (
            f"selected_{item['rank']:02d}_raw_frame_{item['raw_frame_idx']:06d}_boxes.jpg"
        )
        cv2.imwrite(
            str(selected_frame_file),
            cv2.cvtColor(annotated_frame, cv2.COLOR_RGB2BGR),
        )
        time_sec = item['raw_frame_idx'] / source_fps
        ax.set_title(
            f"Selected {item['rank']}/5\n"
            f"input slot={item['feature_slot']} | raw={item['raw_frame_idx']}\n"
            f"t={time_sec:.2f}s | boxes={valid_box_count}"
        )
        ax.axis('off')
        selected_rows.append({
            'selected_rank': item['rank'],
            'input_frame_slot': item['feature_slot'],
            'raw_frame_idx': item['raw_frame_idx'],
            'time_sec': time_sec,
            'selection_weight': item['weight'],
            'labels': ', '.join(map(str, item['labels'][:args.objs])),
        })

    fig.suptitle(
        f"Five model-selected frames with GroundingDINO boxes\n"
        f"{noisy_sample['question_type']}: {noisy_sample['question']}\n"
        f"Annotated answer: {noisy_sample['correct_answer']}",
        fontsize=13,
    )
    plt.tight_layout(rect=[0, 0, 1, 0.86])
    SELECTED_FRAMES_JPG = VIDEO_ARTIFACT_DIR / 'selected_5_frames_with_boxes.jpg'
    fig.savefig(SELECTED_FRAMES_JPG, dpi=250, bbox_inches='tight', facecolor='white')
    plt.show()
    selected_frames_df = pd.DataFrame(selected_rows)
    display(selected_frames_df)
    selected_frames_df.to_csv(VIDEO_ARTIFACT_DIR / 'selected_frames_metadata.csv', index=False)
    print(f'Saved all video-review artifacts to: {VIDEO_ARTIFACT_DIR}')

    # Full playback is optional because rendering many videos makes the notebook heavy.
    if DISPLAY_FULL_VIDEO:
        display(Video(selected_video_path, embed=False, width=800))
    return str(VIDEO_ARTIFACT_DIR)

generated_artifact_dirs = []
for row_index in NOISY_SAMPLE_ROWS:
    try:
        generated_artifact_dirs.append(inspect_noisy_sample(int(row_index)))
    except Exception as error:
        print(f'Failed row {row_index}: {type(error).__name__}: {error}')

print('\nGenerated artifact directories:')
for artifact_dir in generated_artifact_dirs:
    print(f'  - {artifact_dir}')


## Kết quả cần báo cáo

- checkpoint path, state được dùng (`EMA` hoặc model);
- best epoch và best validation `Acc_ALL`;
- prediction của sample, ground truth và confidence;
- bảng/bar chart score test;
- histogram `U`, đường LUM/HUM và mode áp dụng theo question family.
